[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/06-multi-location-comparison.ipynb)

# Multi-Location Comparison

`analyze_multiple_pois` runs the full isochrone → census pipeline for multiple locations in parallel and produces aggregated statistics with optional comparisons. `generate_report` renders the results as an HTML report.

In this notebook you will learn how to:

1. Analyze three cities at once
2. Inspect per-location results
3. Read comparison rankings
4. Compare walk vs drive mode
5. Use a richer variable set
6. Generate an HTML report
7. Use coordinate tuples

## Setup

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

from socialmapper import analyze_multiple_pois, generate_report
from IPython.display import HTML, display

## 1. Three-City Analysis

Analyze Raleigh, Charlotte, and Asheville in North Carolina.

In [ ]:
cities = ["Raleigh, NC", "Charlotte, NC", "Asheville, NC"]

results = analyze_multiple_pois(
    locations=cities,
    travel_time=15,
    travel_mode="drive",
    variables=["population", "median_income"],
)

print(f"Locations analyzed: {len(results['locations'])}")
print(f"Has comparison: {'comparison' in results}")

## 2. Per-Location Results

Each location entry has aggregated stats, block group count, and the raw census data.

In [ ]:
for loc in results["locations"]:
    name = loc["location"]
    bg_count = loc["block_group_count"]
    pop = loc["aggregated"].get("population", {})
    inc = loc["aggregated"].get("median_income", {})
    print(f"\n--- {name} ---")
    print(f"  Block groups:      {bg_count}")
    print(f"  Total population:  {pop.get('total', 0):,.0f}")
    print(f"  Mean pop/BG:       {pop.get('mean', 0):,.0f}")
    print(f"  Mean median income: ${inc.get('mean', 0):,.0f}")

## 3. Comparison Rankings

The `comparison` dict ranks locations by total value for each variable.

In [ ]:
comparison = results["comparison"]

for var, info in comparison.items():
    print(f"\n{var}:")
    print(f"  Highest: {info['highest']}")
    print(f"  Lowest:  {info['lowest']}")
    print(f"  Rankings:")
    for rank in info["ranked"]:
        print(f"    {rank['location']:<20} total={rank['total']:>12,.0f}  mean={rank['mean']:>10,.0f}")

## 4. Walk vs Drive Comparison

Compare reachable demographics between travel modes.

In [ ]:
for mode in ["drive", "walk"]:
    mode_results = analyze_multiple_pois(
        locations=["Raleigh, NC"],
        travel_time=15,
        travel_mode=mode,
        variables=["population"],
        compare=False,
    )
    loc = mode_results["locations"][0]
    pop = loc["aggregated"]["population"]["total"]
    bgs = loc["block_group_count"]
    print(f"{mode:>5}: {pop:>10,.0f} people in {bgs} block groups")

## 5. Richer Variable Set

In [ ]:
rich_results = analyze_multiple_pois(
    locations=cities,
    travel_time=15,
    variables=["population", "median_income", "median_age", "poverty", "housing_units"],
)

# Show comparison for each variable
for var in ["population", "median_income", "poverty"]:
    info = rich_results["comparison"][var]
    print(f"{var}: highest={info['highest']}, lowest={info['lowest']}")

## 6. Generate HTML Report

`generate_report` renders analysis data as a styled HTML document.

In [ ]:
report_html = generate_report(rich_results, format="html")

print(f"Report length: {len(report_html):,} characters")
display(HTML(report_html))

## 7. Coordinate Tuple Input

You can mix city names with `(lat, lon)` tuples.

In [ ]:
mixed = analyze_multiple_pois(
    locations=[
        "Raleigh, NC",
        (35.2271, -80.8431),  # Charlotte coordinates
    ],
    travel_time=10,
    variables=["population"],
)

for loc in mixed["locations"]:
    pop = loc["aggregated"]["population"]["total"]
    print(f"{loc['location']}: {pop:,.0f} people")

## Summary

| What you learned | API |
|---|---|
| Analyze multiple locations at once | `analyze_multiple_pois(locations, ...)` |
| Inspect aggregated stats per location | `result['locations'][i]['aggregated']` |
| Read comparison rankings | `result['comparison'][var]['highest']` |
| Generate HTML reports | `generate_report(data, format='html')` |

**Next notebook:** [07 — Complete Analysis Workflow](07-complete-analysis-workflow.ipynb)